# 第01章 误差的代价——人工复核记录（C组）

> 复核对象：A1-G20 正文（更新版）与 A2-G22 正文。
> 复核方式：逐项核对数值、边界条件、符号口径、实验表与正文结论的一致性；不重复完整推导（推导见另一文件）。
> 结论标记：✔ 通过；△ 需统一口径；✖ 发现不一致（已注明处理建议）。

## 1.1 浮点数与舍入误差

### 1.1.1 格式参数与间距

- ✔ binary32：$p=24$，$u=2^{-24}\approx5.9605\times10^{-8}$，$\epsilon_{\mathrm{mach}}=2^{-23}\approx1.1921\times10^{-7}$。与 A1 表 1.2、A2 表 1.1 一致。
- ✔ binary64：$p=53$，$u=2^{-53}\approx1.1102\times10^{-16}$，$\epsilon_{\mathrm{mach}}=2^{-52}\approx2.2204\times10^{-16}$。NumPy `finfo.eps` 返回后者，A1 与 A2 均正确注明。
- ✔ 最小正规格化数：binary32 $1.1755\times10^{-38}$，binary64 $2.2251\times10^{-308}$；最小正非规格化数 $2^{-149}$、$2^{-1074}$。A1 表 1.2 与格式说明一致。
- ✔ 间距公式 $\Delta_e=2^{e-p+1}$：$p=3$、$[1,2)$ 间距 $1/4$，$[2,4)$ 间距 $1/2$，与 A1 图 1.1 相符。

### 1.1.2 舍入模型与中点取偶

- ✔ 式 (1.7) $|\delta|\le u$：以 $t\in[2^e,2^{e+1})$ 验证，半间距 $2^{e-p}$ 除以 $|t|\ge2^e$ 得 $2^{-p}=u$。
- ✔ 中点取偶例子：$t=1+2^{-53}$ 舍到 1（最低位为 0），$t=1+3\cdot2^{-53}$ 舍到 $1+2^{-52}$；两者相对误差均不超过 $u$。
- △ “机器精度”一词：A1 用 $\epsilon_{\mathrm{mach}}=2^{1-p}$，A2 用 $\varepsilon=2^{1-p}$、$u=2^{-p}$。数学相同，符号不同。**建议**：综合稿统一写 $u$ 与 $\epsilon_{\mathrm{mach}}$，并注明 NumPy `eps` 指 $\epsilon_{\mathrm{mach}}$。

### 1.1.3 有效数字定义

- ✔ 定义 1.2：$\widehat x=\pm a\,10^m$，$1\le a<10$，$|\widehat x-x|\le\frac12 10^{m-n+1}$ 称至少 $n$ 位有效数字。
- ✔ $\pi\approx3.14$：误差 $0.00159265<0.005=\frac12 10^{-2}$，故 3 位；若要求 4 位，右端 $0.0005$，不满足。
- ✔ 式 (1.11)–(1.12)：以 $a_1 10^m\le|\widehat x|<(a_1+1)10^m$ 代入，方向正确；前者必要条件形式、后者充分条件，A1 的措辞准确。
- △ A2 1.1 节未展开有效数字定义。**建议**：保留 A1 定义；若课时紧，可放选读，但不要在正文中混用“显示位数=有效位数”。

### 1.1.4 误差传播与面积例

- ✔ 式 (1.13) 中值定理界：$|f(\widehat x)-f(x)|\le M_1|\widehat x-x|$，严格。
- ✔ 例 1.2：$\widehat l=110$、$\widehat d=80$、误差限 $0.2$、$0.1$。一阶估计 $80\times0.2+110\times0.1=27$；严格界加乘积项 $0.02$ 得 $27.02$。手算复核一致。
- ✔ 式 (1.15) 多元一阶估计：偏导数绝对值求和，符号 $\lesssim$ 表示忽略高阶项。A1 注明“严格界需取上确界”，表述完整。

### 1.1.5 求和次序实验

- ✔ A1 表 1.4：$2^{24}$（float32）与 $2^{53}$（float64）处逐个加 1，增量 0；小量先加得 128；`fsum` 得 128。与“$2^p$ 右侧间距为 2、加 1 落中点舍回”解释一致。
- ✔ A2 1.1 节：$[10^{16},1,-10^{16}]$ 得 0，$[10^{16},-10^{16},1]$ 得 1；float32 与 float64 相同。结论一致：结合律不成立，单次舍入界不能当程序误差界。
- △ 两稿实验输入不同（A1 用 $2^p$ 加 128 个 1，A2 用 $10^{16}$ 三元组）。**建议**：综合稿保留两组，分别说明“中点舍入”与“大数吸收小量”两种机制。

## 1.2 截断误差与灾难性抵消

### 1.2.1 前向差分截断项

- ✔ 式 (1.18)：$D_hf-f'=\frac h2f''(\xi_h)$，$|f''|\le M_2$ 时界 $M_2h/2$。
- ✔ $f=e^x$、$x=1$、$h=10^{-2}$：$D_h\approx2.731918656$，误差约 $0.0136368$；$he/2\approx0.0135914$，$f''(\xi)=e^\xi>e$ 解释略大，量级正确。

### 1.2.2 抵消放大因子

- ✔ 式 (1.19)：分子 $|\Delta a-\Delta b|\le\eta(|a|+|b|)$，除以 $|a-b|$ 得放大因子 $(|a|+|b|)/|a-b|$。
- ✔ 开场 $x=10^{16}$：直接式两次开方误差约 $O(u\sqrt x)$，相减后相对误差量级 $O(ux)\approx1.11$，足以完全失真；有理化式 $O(u)$。A1 式 (1.20) 的量级判断正确。
- ✔ “抵消不制造误差”边界：$1.5-1=0.5$ 可精确表示。A1 与 A2 均强调追溯误差来源，无矛盾。

### 1.2.3 差分步长平衡

- ✔ 式 (1.21)–(1.22)：$E(h)=C_th+C_ru/h$，$h_{\mathrm{bal}}=\sqrt{C_ru/C_t}$；尺度为 1 时 $h_{\mathrm{bal}}\sim\sqrt u\approx1.05\times10^{-8}$。
- ✔ A1 表 1.6：binary64 最小总误差 $6.603\times10^{-9}$，步长约 $10^{-8}$；binary32 最小 $2.352\times10^{-5}$，步长约 $1.585\times10^{-4}$。量级与 $\sqrt u$ 规则相符（binary32 的 $\sqrt u\approx2.44\times10^{-4}$）。
- ✔ A2 表 1.2：最小总绝对误差 $6.60275\times10^{-9}$，$h=10^{-8}$；$h=10^{-16}$ 时取样增量为 0，误差约 $e$。与 A1 结论一致。
- △ 两稿网格不同（A1 151 个对数间隔，A2 16 个十进步长）。**建议**：保留 A1 图展示振荡，A2 表作课堂速查；注明“观测最小点依赖网格”。

### 1.2.4 秦九韶递推

- ✔ 式 (1.24)：$b_0=a_0$，$b_i=xb_{i-1}+a_i$，$b_n=p(x)$；$n$ 次乘、$n$ 次加。
- ✔ 例：$p(x)=2x^4-3x^2+3x-4$，$x=-2$，中间量 $(2,-4,5,-7,10)$，$p(-2)=10$；直接代入 $32-12-6-4=10$。乘法次数逐项连乘 10 次、秦九韶 4 次，与 A1 表 1.7 一致。
- ✔ 求导选读：中间量 $2,-8,21,-49$，$p'(-2)=-49$。对 $p'(x)=8x^3-6x+3$ 代入 $-2$：$-64+12+3=-49$，一致。

### 1.2.5 选读：中心差分与积分成本

- ✔ 中心差分截断界 $M_3h^2/6$；平衡步长 $(Bu/(2A))^{1/3}$。阶数由 $O(h)$ 改为 $O(h^2)$，正确。
- ✔ 积分 $T_n=1/3+1/(6n^2)$：代入 $\sum_{k=1}^{n-1}k^2=(n-1)n(2n-1)/6$ 化简得 $T_n=(2n^2+1)/(6n^2)=1/3+1/(6n^2)$。A1 正文结果正确（中间通分曾出现 $5n^2-3n+1$，属草稿笔误，最终式无误）。
- ✔ $\tau=10^{-6}$：$n\ge\sqrt{1/(6\tau)}\approx408.25$，$n_{\min}=409$，函数求值 410 次；A1 表 1.8 的 $n=408$ 误差 $1.001217\times10^{-6}$（不满足）、$n=409$ 误差 $9.963275\times10^{-7}$（满足），一致。

## 1.3 条件数与稳定性

### 1.3.1 标量条件数

- ✔ 式 (1.27)：$\kappa_f=|xf'/f|$；$f=x^n$ 得 $n$，$f=\log x$ 得 $1/|\log x|$。
- ✔ $x^n$ 例：$n=10$、输入 $2\%$，一阶预测 $20\%$，实际 $1.02^{10}-1\approx21.8994\%$。数值正确。
- ✔ $\log$ 例：$x\to1$ 时 $\kappa\to\infty$；改用 $t=x-1$ 作输入，$\kappa_t\to1$。A1 的“输入变量改变则条件数改变”表述正确。

### 1.3.2 开场问题条件数

- ✔ 式 (1.29)：$\kappa_d=\frac12\sqrt{x/(x+1)}<1/2$，$x\to\infty$ 趋于 $1/2$。
- ✔ $x=10^{16}$：$\kappa_d\approx0.5$，输入相对误差 $u\approx1.11\times10^{-16}$，一阶预测输出相对误差约 $5.55\times10^{-17}$；有理化式实测 $4.592\times10^{-17}$，量级相符。直接式相对误差 1，远超条件数尺度，属算法不稳定。
- ✔ 边界 $x=0$：右差商 $\to-\infty$，相对条件数公式不适用；A1 明确“直接估计函数增量”。
- △ A1 区分“两个独立输入”与“单输入 $x$”两种模型；A2 在 1.3 节也强调“若将两个平方根看作可独立扰动，讨论的是另一个问题”。口径一致，A1 展开更细。**建议**：综合稿保留 A1 的双模型说明，避免读者把“减法条件数”与“$d(x)$ 条件数”混为一谈。

### 1.3.3 稳定性判据与前后向误差

- ✔ 式 (1.30)：$C[1+\kappa_f]u$ 的尺度解释正确；有理化式满足，直接式不满足。
- ✔ 线性方程组残差界：$\|\widehat x-x\|_\infty/\|x\|_\infty\le\kappa_\infty(A)\|r\|_\infty/\|b\|_\infty$。推导方向正确（详见推导文件）。
- ✔ 选读自检“上界不足不能证明实际误差大”：A1 的区分准确。

### 1.3.4 积分递推

- ✔ 范围式 (1.32)：$e^{-1}/(n+1)<I_n<1/(n+1)$。在 $0<x<1$ 上 $e^{-1}<e^{x-1}<1$，乘 $x^n$ 积分，正确。
- ✔ 递推 $I_n=1-nI_{n-1}$：分部积分结果；$I_0=1-e^{-1}\approx0.6321205588$。
- ✔ 向前误差 $\delta_n=(-1)^nn!\delta_0$：$\widehat I_0=0.6321$，$\delta_0\approx-2.0559\times10^{-5}$，$8!\delta_0\approx-0.829$，$\widehat I_8\approx-0.728$，与 A1 表 1.11 一致。
- ✔ 反向误差 $\delta_n=(-1)^{N-n}\frac{n!}{N!}\delta_N$：$\widehat I_9=0.0684$，$\delta_9\approx-0.02321229$，反推到 $n=0$ 除以 $9!=362880$，$|\delta_0|\approx6.397\times10^{-8}$，与表 1.11 反向绝对误差一致。
- ✔ 独立参考式 (1.36)：逐项积分得 $I_n=e^{-1}\sum_{k=0}^\infty1/[k!(n+k+1)]$；尾项界小于 $4.88\times10^{-288}$。参考可靠。
- △ A2 未展开积分递推。**建议**：综合稿保留 A1 为主例，A2 的条件性扰动表作补充。

### 1.3.5 Newton 迭代

- ✔ 式 (1.37)–(1.38)：$e_{k+1}=e_k^2/(2x_k)$；$a=3$、$x_0=2$ 的序列 $2\to1.75\to1.732142857\to1.732050810\to1.7320508075688772$，与 A1 表 1.12 一致。
- ✔ 误差：$x_4$ 误差约 $1.004\times10^{-16}$，停滞；第 3 次更新达 $10^{-8}$，第 4 次达 $10^{-15}$。与正文“事后核对”表述一致。
- ✔ 一般 Newton 二次界 $|e_{k+1}|\le M|e_k|^2/(2m)$：推导方向正确（详见推导文件）。
- △ A2 未展开 Newton。**建议**：保留为选读，突出“收敛速度、有限精度停滞、可计算停止判据”三点。

## 1.4 AI 关联

### 1.4.1 Softmax

- ✔ 式 (1.39)–(1.41)：平移不变性；$m=\max z_j$ 后 $0<e^{z_i-m}\le1$，和 $\in[1,n]$。
- ✔ $z=(1000,1001,1002)$：直接 $e^{1000}$ 在 binary64 上溢（阈值约 709.78），平移后有限；概率约 $(0.0900,0.2447,0.6652)$，和 1。
- ✔ A1 表 1.13：binary32 平移版 $E_\infty=1.234\times10^{-8}$、$E_\Sigma=2.235\times10^{-8}$；binary64 $E_\infty=1.045\times10^{-16}$、$E_\Sigma=1.110\times10^{-16}$。量级与 $u$ 一致。
- ✔ 第 2 组 binary32 平移版 $L\approx1002.40759277$，减 1000 得 $2.40759277$；式 (1.44) 直接算第 1 类损失约 $2.40760612$。A1 的“避免额外抵消”解释正确。

### 1.4.2 交叉熵与尾部概率

- ✔ 式 (1.42)–(1.44)：$\ell=\log\sum e^{z_j-m}-(z_y-m)$。
- ✔ 例 1.7：$z=(0,-1000)$，$y=2$，尾概率约 $5.076\times10^{-435}$，低于 binary64 最小正非规格化数，存 0；先概率再取对数得 $+\infty$，式 (1.44) 得 1000。准确损失 $1000+\log(1+e^{-1000})$。A1 表 1.14 正确。
- ✔ “把零概率截到小常数会改变损失函数”：表述准确。

### 1.4.3 梯度与低精度

- ✔ 式 (1.45)：$\partial\ell/\partial z_j=p_j-\mathbf1_{j=y}$。真实类别 $p_y-1$，非真实 $p_j$。
- ✔ binary16 最小正非规格化数 $2^{-24}$，$10^{-8}$ 小于半间距，舍入为 0；先乘 $1024=2^{10}$ 再存 binary16、在 binary32 中除回，恢复约 $1.001172\times10^{-8}$。数值正确。
- ✔ 更新吸收：$w=1$、学习率 $0.01$、梯度 $10^{-7}$，更新量 $10^{-9}$；binary16 在 1 附近间距 $2^{-10}\approx9.77\times10^{-4}$，binary32 半间距约 $6\times10^{-8}$，$10^{-9}$ 均小于半间距，可能被吸收。A1 区分“梯度下溢”与“更新吸收”，正确。
- △ A2 未展开 1.4 节。**建议**：保留 A1 主稿；Jacobian 界与平移误差展开标选读。

### 1.4.4 梯度差分核验

- ✔ 式 (1.46)：中心差分 $g_i^{(h)}=[\ell(z+he_i)-\ell(z-he_i)]/(2h)$。
- ✔ A1 报告 $10^{-3},10^{-5},10^{-7}$ 中中间步长最大差约 $6.820\times10^{-12}$；类别错写为 1 后逐分量差分检出约 1 的偏差。结论限定“给定输入与实现”，正确。
- △ 该核验为选做。**建议**：综合稿保留为选做，不扩大必修。

## 差异汇总与整合建议

| 项目 | A1 | A2 | 复核结论与建议 |
| --- | --- | --- | --- |
| 机器精度符号 | $u=2^{-p}$，$\epsilon_{\mathrm{mach}}=2^{1-p}$ | $\varepsilon=2^{1-p}$，$u=2^{-p}$ | △ 统一为 $u$ 与 $\epsilon_{\mathrm{mach}}$，注明 NumPy `eps` |
| 有效数字 | 定义 1.2 + 双向不等式 | 未展开 | ✔ 保留 A1，可标选读 |
| 差分扫描 | 151 对数间隔，图 1.2 | 16 十进步长，表 | ✔ 两稿结论一致；保留 A1 图、A2 表 |
| 积分递推 | 例 1.5 完整 + 级数参考 | 未展开 | ✔ 保留 A1 为主例 |
| 条件性实验 | 表 1.9 对数扰动 | $q(x)=1/(1-x)$ 扰动表 | ✔ 互补；A1 更贴开场 |
| Newton | 1.3 节末完整 | 未展开 | ✔ 保留为选读 |
| AI 关联 | 1.4 节完整 | 未展开 | ✔ 保留 A1 主稿，选读标清 |
| 计算代价 | 秦九韶、积分、中心差分 | 未展开 | ✔ 保留 A1 选读，控制深度 |
| 开场漫画/导入 | 无漫画，直接公式 | 有三格漫画 | △ 综合稿可保留 A2 漫画作导入，正文以 A1 公式为主线 |

## 复核结论

- 主要数值表（1.2、1.4、1.6、1.8、1.9、1.11、1.12、1.13、1.14）逐项核对，未发现与正文结论矛盾的数值。
- 主要定理结论（式 1.4、1.7、1.11–1.12、1.19、1.22、1.29、1.34–1.36、1.38、1.41、1.44–1.45）的适用条件与边界在 A1 中均有注明；A2 在未展开处不构成冲突。
- 需统一口径的主要是符号（机器精度）与实验网格表述；建议按上表处理。
- 未发现需要推翻的结论；发现并记录一处草稿通分笔误（积分 $T_n$ 中间式），最终式正确。